# NPIPE A/B vs FFP10 HM1/HM2 noise

Both products are Planck HFI **end-to-end residuals** (simulated sky subtracted): noise plus leftover systematics, no CMB, no Galactic/CIB signal. They are **not** the same split and **not** the same realisation, so maps will not match pixel-to-pixel. They should look similar in **amplitude and angular spectrum** at 100–353 GHz.

| | NPIPE (PR4) | FFP10 (PR3) |
|---|---|---|
| Path | `planck_noise/npipe/` | `planck_noise/half_mission/` |
| Files | `npipe6v20_noise_{ν}_{A\|B}_mc_00200.fits` | `ffp10_noise_{ν}_{hm1\|hm2}_map_mc_00000.fits` |
| Split | detector-set **A / B** (independent horns) | ring-based **half-mission 1 / 2** |
| Realisation on disk | `mc_00200` | `mc_00000` |
| \(N_\mathrm{side}\) | 2048, RING, Galactic | 2048, RING, Galactic |
| 100–353 | treat as \(\mathrm{K}_\mathrm{CMB}\) (\(\times 10^6\)) | `TUNIT1=K_CMB` (\(\times 10^6\)) |
| 545/857 | treat as \(\mathrm{MJy\,sr^{-1}}\); **Plik** \(U_c\) | `TUNIT1=MJy/sr`; **Plik** \(U_c\) |

Null tests (independent halves \(\Rightarrow\) small cross): NPIPE **A \(\times\) B** vs FFP10 **HM1 \(\times\) HM2**.

Units: Planck 2018 V \(U_c(545)=58.062295\), \(U_c(857)=2.2703657\) MJy/sr per \(K_\mathrm{CMB}\). At 100–143 GHz the two products track. From 217 GHz up, NPIPE A/B retain more Galactic-plane residual (gain/ADC leakage of dust) than FFP10 half-missions; at 545 GHz FFP10 is ~20× noisier in RMS, at 857 GHz NPIPE is ~5× noisier. Those are pipeline differences, not unit bugs.


## 1. Configuration


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import healpy as hp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

_REPO = Path("/scratch/scratch-lxu/flamingo_mock_analysis")
sys.path.insert(0, str(_REPO / "scripts"))

from flamingo_mock.powerspectra import bin_cl, compute_cl  # noqa: E402
from pub_style import apply_pub_style, no_grid, savefig  # noqa: E402

apply_pub_style()
mpl.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

FREQUENCIES = (100, 143, 217, 353, 545, 857)
NSIDE = 2048
LMAX = 2 * NSIDE
DELTA_ELL = 21
RECOMPUTE_CL = False

NOISE_ROOT = Path("/rds/rds-lxu/flamingo/integrated_maps_synthetic") / "planck_noise"
NPIPE_ROOT = NOISE_ROOT / "npipe"
FFP_ROOT = NOISE_ROOT / "half_mission"
FIG_DIR = _REPO / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

NPIPE_REAL = 200
FFP_REAL = 0
UC_MJY_PER_KCMB = {545: 58.062295, 857: 2.2703657}

NPIPE_CL = NPIPE_ROOT / f"spectra_npipe_ab_mc{NPIPE_REAL:05d}_plik.npz"
FFP_CL = FFP_ROOT / f"spectra_ffp10_hm_mc{FFP_REAL:05d}_plik.npz"

print(f"npipe : {NPIPE_ROOT}  mc_{NPIPE_REAL:05d}  A/B")
print(f"ffp10 : {FFP_ROOT}  mc_{FFP_REAL:05d}  hm1/hm2")
print(f"lmax  : {LMAX}")
print(f"cache : {NPIPE_CL.name}  |  {FFP_CL.name}")


## 2. Paths


In [ ]:
def npipe_path(freq: int, split: str) -> Path:
    return NPIPE_ROOT / f"{freq}GHz" / split / f"npipe6v20_noise_{freq}_{split}_mc_{NPIPE_REAL:05d}.fits"


def ffp_path(freq: int, split: str) -> Path:
    return FFP_ROOT / f"{freq}GHz" / split / f"ffp10_noise_{freq}_{split}_map_mc_{FFP_REAL:05d}.fits"


print(f"{'product':<8} {'nu':>5} {'split':>4}  {'MB':>6}  path")
for freq in FREQUENCIES:
    for split, fn in (("A", npipe_path), ("B", npipe_path), ("hm1", ffp_path), ("hm2", ffp_path)):
        p = fn(freq, split)
        ok = p.is_file() and p.stat().st_size > 1_000_000
        mb = p.stat().st_size / 1e6 if p.is_file() else 0.0
        tag = "npipe" if split in ("A", "B") else "ffp10"
        print(f"{tag:<8} {freq:5d} {split:>4}  {mb:6.0f}  {'ok' if ok else 'MISSING'}  {p.name}")


## 3. Load in \(\mu\mathrm{K}_\mathrm{CMB}\)

100–353 GHz: \(\times 10^6\). 545/857 GHz: Plik \(U_c\). NPIPE 857 B has one UNSEEN-scale pixel; it is set to zero. Same sanitising for any non-finite / \(|T|>10^{20}\) value.


In [ ]:
def to_uK(m_native: np.ndarray, freq: int) -> np.ndarray:
    m = np.asarray(m_native, dtype=np.float64)
    if freq <= 353:
        return m * 1.0e6
    return m / UC_MJY_PER_KCMB[int(freq)] * 1.0e6


def sanitize(m: np.ndarray) -> np.ndarray:
    bad = ~np.isfinite(m) | (np.abs(m) > 1.0e20)
    if bad.any():
        m = m.copy()
        m[bad] = 0.0
    return m


def load_npipe(freq: int, split: str) -> np.ndarray:
    return sanitize(to_uK(hp.read_map(str(npipe_path(freq, split)), field=0, dtype=np.float64), freq))


def load_ffp(freq: int, split: str) -> np.ndarray:
    return sanitize(to_uK(hp.read_map(str(ffp_path(freq, split)), field=0, dtype=np.float64), freq))


print(f"{'nu':>5}  {'NPIPE A':>10}  {'NPIPE B':>10}  {'FFP HM1':>10}  {'FFP HM2':>10}  {'A/HM1':>8}  {'B/HM2':>8}")
print(f"{'GHz':>5}  {'uK':>10}  {'uK':>10}  {'uK':>10}  {'uK':>10}")
rms = {}
for freq in FREQUENCIES:
    maps = {
        "npipe_A": load_npipe(freq, "A"),
        "npipe_B": load_npipe(freq, "B"),
        "ffp_hm1": load_ffp(freq, "hm1"),
        "ffp_hm2": load_ffp(freq, "hm2"),
    }
    s = {k: float(np.std(v)) for k, v in maps.items()}
    rms[freq] = s
    print(
        f"{freq:5d}  {s['npipe_A']:10.3f}  {s['npipe_B']:10.3f}  "
        f"{s['ffp_hm1']:10.3f}  {s['ffp_hm2']:10.3f}  "
        f"{s['npipe_A']/s['ffp_hm1']:8.3f}  {s['npipe_B']/s['ffp_hm2']:8.3f}"
    )
    del maps


### Scan anisotropy

Planck rings about the ecliptic axis: ecliptic-plane RMS / ecliptic-pole RMS should be \(\sim 1.6\) for detector noise. Galactic plane / pole \(\approx 1\) means no sky signal in the residual. \(\beta\) = ecliptic latitude.


In [ ]:
def region_rms(m: np.ndarray) -> dict[str, float]:
    nside = hp.get_nside(m)
    th, ph = hp.pix2ang(nside, np.arange(m.size))
    b = np.pi / 2.0 - th
    # ecliptic latitude of Galactic (theta,phi)
    r = hp.Rotator(coord=["G", "E"])
    th_e, _ = r(th, ph)
    beta = np.pi / 2.0 - th_e
    return {
        "gal_plane": float(np.std(m[np.abs(b) < np.deg2rad(10.0)])),
        "gal_pole": float(np.std(m[np.abs(b) > np.deg2rad(60.0)])),
        "ecl_plane": float(np.std(m[np.abs(beta) < np.deg2rad(15.0)])),
        "ecl_pole": float(np.std(m[np.abs(beta) > np.deg2rad(60.0)])),
    }


print(f"{'nu':>5}  {'prod':>8}  {'gal pl/po':>9}  {'ecl pl/po':>9}")
for freq in FREQUENCIES:
    for name, loader, split in (
        ("npipe A", load_npipe, "A"),
        ("ffp HM1", load_ffp, "hm1"),
    ):
        m = loader(freq, split)
        r = region_rms(m)
        print(
            f"{freq:5d}  {name:>8}  {r['gal_plane']/r['gal_pole']:9.2f}  "
            f"{r['ecl_plane']/r['ecl_pole']:9.2f}"
        )
        del m


## 4. Mollweide: NPIPE A vs FFP10 HM1

Same colour scale per frequency (1st–99th percentile of the **two** maps together). Independent realisations \(\Rightarrow\) stripes align with the scan, not with each other. If one panel looks blank (545 GHz NPIPE), that map’s RMS is much smaller than the other — that *is* the comparison.


In [ ]:
fig = plt.figure(figsize=(12.4, 14.8))
for i, freq in enumerate(FREQUENCIES):
    a = load_npipe(freq, "A")
    b = load_ffp(freq, "hm1")
    lo, hi = np.percentile(np.concatenate([a, b]), [1.0, 99.0])
    hp.mollview(
        a,
        fig=fig.number,
        sub=(6, 2, 2 * i + 1),
        title=rf"NPIPE A  ${freq:g}\,\mathrm{{GHz}}$",
        unit=r"$\mu\mathrm{K}_\mathrm{CMB}$",
        min=lo,
        max=hi,
        cmap="RdBu_r",
        notext=True,
        cbar=True,
    )
    hp.mollview(
        b,
        fig=fig.number,
        sub=(6, 2, 2 * i + 2),
        title=rf"FFP10 HM1  ${freq:g}\,\mathrm{{GHz}}$",
        unit=r"$\mu\mathrm{K}_\mathrm{CMB}$",
        min=lo,
        max=hi,
        cmap="RdBu_r",
        notext=True,
        cbar=True,
    )
    del a, b
fig.suptitle(r"Noise residual, shared colour scale per frequency  ($\mathrm{mc\,00200}$ vs $\mathrm{mc\,00000}$)", y=1.01, fontsize=14)
no_grid()
savefig(fig, "npipe_vs_ffp10_noise_mollweide", fig_dir=FIG_DIR)
plt.show()


## 5. Auto-spectra

Cached under the product directories. Set `RECOMPUTE_CL = True` to rebuild. FFP10 cache is the Plik-unit file from `visualize_ffp10_hm_noise.ipynb`. Pixel window is **not** deconvolved (same as that notebook).


In [ ]:
def load_or_compute(cache: Path, kind: str) -> dict:
    need = [f"cl_{s}_{nu}" for nu in FREQUENCIES for s in ("A", "B", "x")] if kind == "npipe" else [
        f"cl_{s}_{nu}" for nu in FREQUENCIES for s in ("hm1", "hm2", "x")
    ]
    if cache.is_file() and not RECOMPUTE_CL:
        z = np.load(cache)
        if all(k in z.files for k in need):
            print("loaded", cache)
            return {k: z[k] for k in z.files}
    cls = {"ell": np.arange(LMAX + 1, dtype=np.float64)}
    for nu in FREQUENCIES:
        print(f"anafast {kind} {nu} GHz ...", flush=True)
        if kind == "npipe":
            a, b = load_npipe(nu, "A"), load_npipe(nu, "B")
            cls[f"cl_A_{nu}"] = compute_cl(a, lmax=LMAX, deconv_pixel_window=False)
            cls[f"cl_B_{nu}"] = compute_cl(b, lmax=LMAX, deconv_pixel_window=False)
            cls[f"cl_x_{nu}"] = compute_cl(a, b, lmax=LMAX, deconv_pixel_window=False)
        else:
            a, b = load_ffp(nu, "hm1"), load_ffp(nu, "hm2")
            cls[f"cl_hm1_{nu}"] = compute_cl(a, lmax=LMAX, deconv_pixel_window=False)
            cls[f"cl_hm2_{nu}"] = compute_cl(b, lmax=LMAX, deconv_pixel_window=False)
            cls[f"cl_x_{nu}"] = compute_cl(a, b, lmax=LMAX, deconv_pixel_window=False)
        del a, b
    np.savez(cache, **cls)
    print("wrote", cache)
    return cls


npipe_cls = load_or_compute(NPIPE_CL, "npipe")
ffp_cls = load_or_compute(FFP_CL, "ffp")


### Per-frequency \(C_\ell\)

Solid: NPIPE A, B. Dashed: FFP10 HM1, HM2. Dotted: signed cross (symlog) for each product’s null.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12.4, 7.4), sharex=True)
axes = axes.ravel()
for ax, nu in zip(axes, FREQUENCIES):
    e, cA = bin_cl(npipe_cls[f"cl_A_{nu}"], delta_ell=DELTA_ELL, lmin=2)
    _, cB = bin_cl(npipe_cls[f"cl_B_{nu}"], delta_ell=DELTA_ELL, lmin=2)
    _, c1 = bin_cl(ffp_cls[f"cl_hm1_{nu}"], delta_ell=DELTA_ELL, lmin=2)
    _, c2 = bin_cl(ffp_cls[f"cl_hm2_{nu}"], delta_ell=DELTA_ELL, lmin=2)
    ax.loglog(e, cA, color="C0", lw=1.7, label=r"NPIPE A")
    ax.loglog(e, cB, color="C0", lw=1.2, ls=":", label=r"NPIPE B")
    ax.loglog(e, c1, color="C3", lw=1.5, ls="--", label=r"FFP10 HM1")
    ax.loglog(e, c2, color="C3", lw=1.1, ls="-.", label=r"FFP10 HM2")
    ax.set_title(rf"{nu:g}\,GHz")
    ax.set_xlim(2, LMAX)
    ax.set_ylabel(r"$C_\ell$ [$\mu\mathrm{K}^2$]")
    no_grid(ax)
axes[0].legend(loc="lower left", fontsize=8)
for ax in axes[3:]:
    ax.set_xlabel(r"Multipole $\ell$")
fig.suptitle(r"NPIPE A/B vs FFP10 HM1/HM2 noise auto-spectra", y=1.01)
fig.tight_layout()
savefig(fig, "npipe_vs_ffp10_noise_cl_auto", fig_dir=FIG_DIR)
plt.show()


### Null test: A \(\times\) B vs HM1 \(\times\) HM2

\(r_\ell = C_\ell^{\mathrm{cross}} / \sqrt{C_\ell^{(1)} C_\ell^{(2)}}\). Independent detector sets (NPIPE) should sit near 0 at 100–217 GHz. Half-missions share scan/destriper residuals at low \(\ell\). At 353/545 NPIPE A and B can correlate because the residual still contains **signal-proportional** systematics (both sets see the same dust).


In [ ]:
def r_of(c1, c2, cx):
    e1, a = bin_cl(c1, delta_ell=DELTA_ELL, lmin=2)
    _, b = bin_cl(c2, delta_ell=DELTA_ELL, lmin=2)
    _, x = bin_cl(cx, delta_ell=DELTA_ELL, lmin=2)
    den = np.sqrt(np.clip(a, 0, None) * np.clip(b, 0, None))
    r = np.full_like(x, np.nan)
    good = den > 0
    r[good] = x[good] / den[good]
    return e1, r


fig, axes = plt.subplots(2, 3, figsize=(12.4, 7.0), sharex=True, sharey=True)
axes = axes.ravel()
for ax, nu in zip(axes, FREQUENCIES):
    e, r_n = r_of(npipe_cls[f"cl_A_{nu}"], npipe_cls[f"cl_B_{nu}"], npipe_cls[f"cl_x_{nu}"])
    _, r_f = r_of(ffp_cls[f"cl_hm1_{nu}"], ffp_cls[f"cl_hm2_{nu}"], ffp_cls[f"cl_x_{nu}"])
    ax.plot(e, r_n, color="C0", lw=1.5, label=r"NPIPE $A\times B$")
    ax.plot(e, r_f, color="C3", lw=1.3, ls="--", label=r"FFP10 HM1$\times$HM2")
    ax.axhline(0.0, color="k", lw=0.7)
    ax.set_xscale("log")
    ax.set_xlim(2, LMAX)
    ax.set_ylim(-0.4, 1.05)
    ax.set_title(rf"{nu:g}\,GHz")
    no_grid(ax)
axes[0].legend(loc="upper right", fontsize=8)
for ax in axes[3:]:
    ax.set_xlabel(r"Multipole $\ell$")
for ax in axes[::3]:
    ax.set_ylabel(r"$r_\ell$")
fig.suptitle(r"Split correlation: detector sets vs half-missions", y=1.01)
fig.tight_layout()
savefig(fig, "npipe_vs_ffp10_noise_r_ell", fig_dir=FIG_DIR)
plt.show()


### High-\(\ell\) summary

Mean \(C_\ell\) over \(500\le\ell\le 2000\). Ratio \(\langle C\rangle_\mathrm{NPIPE\,A}/\langle C\rangle_\mathrm{FFP10\,HM1}\) is the squared RMS ratio if both were white; the table also prints that RMS ratio.


In [ ]:
ell = np.arange(LMAX + 1)
band = (ell >= 500) & (ell <= 2000)
print(f"{'nu':>5}  {'<C> NPIPE A':>12}  {'<C> FFP HM1':>12}  {'C ratio':>8}  {'rms A/HM1':>10}  {'<r> AxB':>8}  {'<r> HM':>8}")
print(f"{'GHz':>5}  {'uK^2':>12}  {'uK^2':>12}")
for nu in FREQUENCIES:
    cA = npipe_cls[f"cl_A_{nu}"][: LMAX + 1]
    c1 = ffp_cls[f"cl_hm1_{nu}"][: LMAX + 1]
    cxn = npipe_cls[f"cl_x_{nu}"][: LMAX + 1]
    cxf = ffp_cls[f"cl_x_{nu}"][: LMAX + 1]
    cB = npipe_cls[f"cl_B_{nu}"][: LMAX + 1]
    c2 = ffp_cls[f"cl_hm2_{nu}"][: LMAX + 1]
    mA, m1 = float(np.nanmean(cA[band])), float(np.nanmean(c1[band]))
    rn = float(np.nanmean(cxn[band] / np.sqrt(np.clip(cA[band], 0, None) * np.clip(cB[band], 0, None))))
    rf = float(np.nanmean(cxf[band] / np.sqrt(np.clip(c1[band], 0, None) * np.clip(c2[band], 0, None))))
    print(
        f"{nu:5d}  {mA:12.4e}  {m1:12.4e}  {mA/m1:8.3f}  "
        f"{rms[nu]['npipe_A']/rms[nu]['ffp_hm1']:10.3f}  {rn:8.3f}  {rf:8.3f}"
    )
